# Day 15: Time Series Analysis

## Creating a Time Series
A time series is just data indexed by time instead of by row number — sales per day, stock prices per minute, sensor readings per hour, and so on. pandas has dedicated support for this: a `DatetimeIndex` lets you slice by date range, resample to a different frequency, and compute rolling statistics, all of which would be tedious with plain lists or arrays. In the code below, `pd.date_range` generates a sequence of evenly spaced dates, and we attach that as the index of a `pd.Series` holding some values — this is the standard first step before doing any time-based analysis in pandas. When you print the result, notice the dates are now the row labels instead of 0, 1, 2, ... — that's what makes date-aware operations possible.</cell id="cell-1">


In [ ]:
import pandas as pd
import numpy as np

# generate 30 consecutive daily dates starting Jan 1, 2024
dates = pd.date_range(start='2024-01-01', periods=30, freq='D')
values = np.random.randint(50, 150, size=30)
# use the dates as the index instead of default integer positions -> this makes it a proper time series
ts = pd.Series(values, index=dates)
print(ts.head())

## Resampling
Raw data often comes at a finer time grain than you need for analysis — you might have daily sales but want to report weekly or monthly totals. Resampling groups your time series into fixed time buckets (like 'W' for weekly, 'M' for monthly) and then applies an aggregation function to each bucket, similar to how `groupby` aggregates by category instead of by time. This is a common step in reporting and dashboarding, where raw granular data needs to be rolled up to match the audience's needs. The output below is a shorter series — one row per week — where each value is the average of that week's daily values, with the index now showing week-ending dates.</cell id="cell-3">


In [ ]:
# resample daily data into weekly buckets ('W') and average the values in each week
print(ts.resample('W').mean())

## Rolling / Moving Average
Day-to-day data is often noisy — a single unusually high or low day can obscure the bigger picture. A rolling (moving) average fixes this by, at each point, averaging that point together with a fixed number of previous points (the "window"), which smooths out random noise and makes the underlying trend easier to see. This is widely used in finance (moving averages of stock prices), sales analysis, and monitoring dashboards. In the output, notice the first `window - 1` values are `NaN` — there aren't enough prior days yet to fill a full 7-day window, so pandas can't compute an average until day 7.</cell id="cell-5">


In [ ]:
# rolling(window=7) looks at each point plus the 6 before it, then .mean() averages that 7-day window
rolling_avg = ts.rolling(window=7).mean()
print(rolling_avg.head(10))  # first 6 values are NaN - not enough prior days to fill the window yet

## Visualizing Trend
Plotting the raw series next to its rolling average makes the smoothing effect concrete — it's a quick, visual way to communicate "here's the noisy data, and here's the underlying trend" to anyone reviewing the analysis. This kind of chart is a staple in reports and dashboards for exactly that reason: it lets a viewer separate day-to-day noise from the real direction the numbers are moving. When you look at the plot, expect the "Daily" line to be jagged/spiky while the "7-day Rolling Avg" line is a smoother curve that lags slightly behind sudden changes (since it's averaging in the past).</cell id="cell-7">


In [ ]:
import matplotlib.pyplot as plt

plt.plot(ts, label='Daily')  # raw, noisy daily values
plt.plot(rolling_avg, label='7-day Rolling Avg')  # smoothed trend line
plt.legend()
plt.title('Time Series Trend')
plt.show()